In [1]:
import importlib
import numpy as np
import polars as pl
import scipy.sparse as sp
import torch
import torch.nn as nn
from tqdm import tqdm

from datasets import DATA_FOLDER, Dataloader, prepare_interaction_data, split_input_target_interactions
from teaser import load_item_tag_matrix
from util import CHECKPOINT_FOLDER, get_checkpoint_filepath, load_checkpoint, load_config_from_checkpoint, set_seed

#device = torch.device("cuda") if torch.cuda.is_available() else torch.device("mps") if torch.mps.is_available() else torch.device("cpu")
device = torch.device("cpu")

MIN_USER_TAG_SUPPORT = 5
MAX_USER_TAG_SUPPORT = 100
DATASET = "MSD"
ELSA_CHECKPOINT_PATH = f"{CHECKPOINT_FOLDER}/{DATASET}/TopKSAE-8192-26b50839.ckpt"  # ELSA + L2
MULTVAE_CHECKPOINT_PATH = f"{CHECKPOINT_FOLDER}/{DATASET}/TopKSAE-8192-369a6053.ckpt"  # MultVAE + L2
TEASER_CHECKPOINT_PATH = f"{CHECKPOINT_FOLDER}/{DATASET}/TEASERGD-1753-51af622f.ckpt"  # TEASER-GD


elsa_sae_cfg = load_config_from_checkpoint(ELSA_CHECKPOINT_PATH)
elsa_checkpoint_path = f"{CHECKPOINT_FOLDER}/{DATASET}/{elsa_sae_cfg['pretrained_model_checkpoint']}"
elsa_cfg = load_config_from_checkpoint(elsa_checkpoint_path)
multvae_sae_cfg = load_config_from_checkpoint(MULTVAE_CHECKPOINT_PATH)
multvae_checkpoint_path = f"{CHECKPOINT_FOLDER}/{DATASET}/{multvae_sae_cfg['pretrained_model_checkpoint']}"
multvae_cfg = load_config_from_checkpoint(multvae_checkpoint_path)
teaser_cfg = load_config_from_checkpoint(TEASER_CHECKPOINT_PATH)

set_seed(elsa_cfg["seed"])

interactions_df, train_csr, val_csr, test_csr, train_users, val_users, test_users, items = prepare_interaction_data(
    elsa_cfg
)  # both models use the same dataset configuration
train_users_to_idxs = {uid: uidx for uidx, uid in enumerate(train_users)}
val_users_to_idxs = {uid: uidx for uidx, uid in enumerate(val_users)}
test_users_to_idxs = {uid: uidx for uidx, uid in enumerate(test_users)}
items_to_idxs = {iid: iidx for iidx, iid in enumerate(items)}

items_df = (
    pl.scan_csv(f"{DATA_FOLDER}/{DATASET}/song_names.csv").rename({"song_id": "item_id"}).cast({"item_id": pl.String}).cast({"item_id": pl.Categorical}).collect()
)


elsa_model_class = getattr(importlib.import_module(elsa_cfg["model_module"]), elsa_cfg["model_class"])
elsa_model = elsa_model_class(train_csr.shape[1], elsa_cfg["embedding_dim"]).to(device)
multvae_model_class = getattr(importlib.import_module(multvae_cfg["model_module"]), multvae_cfg["model_class"])
multvae_model = multvae_model_class(
    train_csr.shape[1],
    [int(x) for x in multvae_cfg["hidden_dims"].split(",") if x.strip()],
    multvae_cfg["embedding_dim"],
    multvae_cfg["annealing_beta"],
    multvae_cfg["annealing_steps"],
).to(device)
_, _ = load_checkpoint(elsa_model, None, get_checkpoint_filepath(elsa_cfg), device, None)
_, _ = load_checkpoint(multvae_model, None, get_checkpoint_filepath(multvae_cfg), device, None)

elsa_sae_model_class = getattr(importlib.import_module(elsa_sae_cfg["model_module"]), elsa_sae_cfg["model_class"])
elsa_sae = elsa_sae_model_class(
    elsa_cfg["embedding_dim"],
    elsa_sae_cfg["embedding_dim"],
    elsa_sae_cfg["reconstruction_loss"],
    l1_coef=elsa_sae_cfg["l1_coef"],
    k=elsa_sae_cfg["k"],
).to(device)
multvae_sae_model_class = getattr(importlib.import_module(multvae_sae_cfg["model_module"]), multvae_sae_cfg["model_class"])
multvae_sae = multvae_sae_model_class(
    multvae_cfg["embedding_dim"],
    multvae_sae_cfg["embedding_dim"],
    multvae_sae_cfg["reconstruction_loss"],
    l1_coef=multvae_sae_cfg["l1_coef"],
    k=multvae_sae_cfg["k"],
).to(device)
_, _ = load_checkpoint(elsa_sae, None, get_checkpoint_filepath(elsa_sae_cfg), device, None)
_, _ = load_checkpoint(multvae_sae, None, get_checkpoint_filepath(multvae_sae_cfg), device, None)

teaser_model_class = getattr(importlib.import_module(teaser_cfg["model_module"]), teaser_cfg["model_class"])
teaser_item_tag_matrix, teaser_tags = load_item_tag_matrix(teaser_cfg["dataset"], items, teaser_cfg["tag_min_count"], weighted=False)
teaser_model = teaser_model_class(
    teaser_item_tag_matrix.to(device),
    lambda1=teaser_cfg["lambda1"],
    lambda2=teaser_cfg["lambda2"],
    init_std=teaser_cfg["init_std"],
).to(device)
_, _ = load_checkpoint(teaser_model, None, get_checkpoint_filepath(teaser_cfg), device, None)

cf_models = {"ELSA": elsa_model, "MultVAE": multvae_model}
sae_models = {"ELSA": elsa_sae, "MultVAE": multvae_sae}
direct_models = {"TEASER": teaser_model}

Removing items with < 200 interactions...
Removing users with < 20 interactions...
Dataset info: users=571355, items=41140, interactions=33633450
Train split info: users=457084, items=41140, interactions=26896271
Val split info: users=57136, items=41140, interactions=3383422
Test split info: users=57135, items=41140, interactions=3353757
Loaded checkpoint from checkpoints/MSD/ELSA-1024-b9aa5195.ckpt (after 25 epochs)
Loaded checkpoint from checkpoints/MSD/MultVAE-1024-da1abc6f.ckpt (after 25 epochs)
Loaded checkpoint from checkpoints/MSD/TopKSAE-8192-26b50839.ckpt (after 97 epochs)
Loaded checkpoint from checkpoints/MSD/TopKSAE-8192-369a6053.ckpt (after 999 epochs)
Loaded checkpoint from checkpoints/MSD/TEASERGD-1753-51af622f.ckpt (after 24 epochs)


In [2]:
from elsa import ELSA


sparse_item_embedding_dict = {}

for base_model_name in cf_models:
    print(base_model_name)
    sparse_item_embeddings = []
    onehot_items_dataloader = Dataloader(sp.eye(len(items), dtype=np.float32, format="csr"), batch_size=1024, device=device)
    with torch.no_grad():
        for onehot_batch in tqdm(onehot_items_dataloader):
            cf_model = cf_models[base_model_name]
            user_embedding = cf_model.encode(onehot_batch)  # Tensor, shape = (batch.shape[0] x elsa_cfg["embedding_dim"])
            if cf_model.__class__ != ELSA:
                user_embedding = user_embedding[0]
            sae = sae_models[base_model_name]
            sae_embedding, _, input_mean, input_std = sae.encode(user_embedding)
            sparse_item_embeddings.append(sp.csr_matrix(sae_embedding.cpu().numpy()))
    sparse_item_embeddings = sp.vstack(sparse_item_embeddings)
    sparse_item_embedding_dict[base_model_name] = sparse_item_embeddings

    neuron_is_alive = np.asarray(sparse_item_embeddings.sum(axis=0)).flatten() != 0
    living_neurons = np.where(neuron_is_alive)[0]
    dead_neuron_count = sparse_item_embeddings.shape[1] - neuron_is_alive.sum()
    print(f"{dead_neuron_count} dead neurons out of {sparse_item_embeddings.shape[1]} ({dead_neuron_count / sparse_item_embeddings.shape[1]:.2%})")
    print(f"{neuron_is_alive.sum()} alive ones\n")

ELSA


100%|██████████| 41/41 [00:06<00:00,  6.70it/s]


1730 dead neurons out of 8192 (21.12%)
6462 alive ones

MultVAE


100%|██████████| 41/41 [00:09<00:00,  4.43it/s]

5806 dead neurons out of 8192 (70.87%)
2386 alive ones



In [3]:
tag_df = (
    pl.scan_csv(f"{DATA_FOLDER}/{DATASET}/song_tag_assignment.csv")
    .rename({"song_id": "item_id"})
    .cast({"tag": pl.String, "item_id": pl.String})
    .cast({"tag": pl.Categorical, "item_id": pl.Categorical})
    .filter(pl.col("tag").count().over("tag") >= 100)  # keep only tags assigned at least 100 times
    .filter(pl.col("item_id").is_in(items))  # keep only items with interactions
    .with_columns(pl.col("item_id").replace_strict(items_to_idxs).alias("item_idx"))
    .collect()
)

tag_df

/tmp/ipykernel_550078/2835131377.py:9: CategoricalRemappingWarning: Local categoricals have different encodings, expensive re-encoding is done to perform this merge operation. Consider using a StringCache or an Enum type if the categories are known in advance
  .collect()


item_id,tag,weight,item_idx
cat,cat,f64,i64
"""SONCXZL12A8C13F6B9""","""classic rock""",1.0,11262
"""SOZMEWR12A6701EBD9""","""classic rock""",1.0,11631
"""SOBUOEK12A6D4F9908""","""classic rock""",1.0,4906
"""SONYAFR12AB017F05A""","""classic rock""",1.0,13598
"""SOQJHUW12AB0188A24""","""classic rock""",1.0,7979
…,…,…,…
"""SOCVHZE12AB01825AB""","""outskirts of expansion""",33.0,21814
"""SOBEMUS12AB0183B42""","""outskirts of expansion""",50.0,21865
"""SOHKXSO12AB0189B61""","""outskirts of expansion""",100.0,26649


In [4]:
def get_tag_item_counts(tag_df: pl.DataFrame, user_subset = None) -> tuple[np.ndarray, sp.csr_matrix]:
    #tags = tag_df["tag"].unique(maintain_order=True).to_numpy()
    tags = tag_df["tag"].cat.get_categories().to_numpy()
    num_tags = len(tags)
    num_items = len(items)
    #if user_subset is not None: #not known for MSD data
    #    tag_df = tag_df.filter(pl.col("user_id").is_in(user_subset))
    row_idx = tag_df["tag"].to_physical().to_numpy()
    
    col_idx = tag_df["item_idx"].to_numpy()
    data    = tag_df["weight"].cast(pl.Float32).to_numpy()
    
    tag_item_counts = sp.csr_matrix(
        (data, (row_idx, col_idx)),
        shape=(num_tags, num_items),
    )
    return tags, tag_item_counts


tags, train_tag_item_counts = get_tag_item_counts(tag_df, user_subset=train_users)

tags, train_tag_item_counts

(array(['classic rock', 'Progressive rock', 'blues', ...,
        'outskirts of expansion', 'Michael Stanley Band', 'huumoria'],
       shape=(1761,), dtype=object),
 <Compressed Sparse Row sparse matrix of dtype 'float32'
 	with 692080 stored elements and shape (1761, 41140)>)

In [5]:
from sae import SAE


class SAESteeredModel:
    def __init__(self, base_model, sae: SAE, concept_neuron_mapping: torch.Tensor, alpha: float):
        self.base_model = base_model
        self.sae = sae
        self.concept_neuron_mapping = concept_neuron_mapping
        self.alpha = alpha  # steering strength

    def eval(self):
        self.base_model.eval()

    @torch.no_grad()
    def encode(self, interaction_batch: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        user_embeddings = self.base_model.encode(interaction_batch)
        if self.base_model.__class__ != ELSA:
            user_embeddings = user_embeddings[0]
        sae_embeddings, _, input_mean, input_std = self.sae.encode(user_embeddings)
        return sae_embeddings, input_mean, input_std

    @torch.no_grad()
    def decode(self, sae_embeddings: torch.Tensor, input_mean: torch.Tensor, input_std: torch.Tensor) -> torch.Tensor:
        return self.base_model.decode(self.sae.decode(sae_embeddings, input_mean, input_std))

    @torch.no_grad()
    def recommend(self, batch: tuple[torch.Tensor, torch.Tensor], k: int, mask_interactions: bool = True) -> tuple[torch.Tensor, torch.Tensor]:
        interaction_batch, steering_concept_id_batch = batch

        sae_embeddings, input_mean, input_std = self.encode(interaction_batch)
        neuron_batch = self.concept_neuron_mapping[steering_concept_id_batch]
        s = sae_embeddings.sum(-1, keepdim=True)
        sae_embeddings *= (1 - self.alpha) / s
        sae_embeddings[torch.arange(neuron_batch.shape[0]), neuron_batch] += self.alpha
        sae_embeddings *= s
        if self.base_model.__class__ != ELSA:
            scores = self.decode(sae_embeddings, input_mean, input_std)
        else:
            scores = nn.ReLU()(self.decode(sae_embeddings, input_mean, input_std) - interaction_batch)

        if mask_interactions:
            scores = torch.where(interaction_batch != 0, 0, scores)  # mask input interactions
        topk_scores, topk_indices = torch.topk(scores, k)
        return topk_scores.cpu().numpy(), topk_indices.cpu().numpy()




class TEASERSteeredModel:
    def __init__(self, base_model, alpha: float):
        self.base_model = base_model
        self.alpha = alpha

    def eval(self):
        self.base_model.eval()

    @torch.no_grad()
    def recommend(self, batch: tuple[torch.Tensor, torch.Tensor], k: int, mask_interactions: bool = True):
        interaction_batch, steering_concept_id_batch = batch
        profiles = self.base_model.user_profiles(interaction_batch)
        feedback = torch.zeros_like(profiles)
        feedback[torch.arange(feedback.shape[0]), steering_concept_id_batch] = self.alpha
        scores, _ = self.base_model.steer_scores(interaction_batch, feedback)
        if mask_interactions:
            scores = torch.where(interaction_batch != 0, 0, scores)
        topk_scores, topk_indices = torch.topk(scores, k)
        return topk_scores.cpu().numpy(), topk_indices.cpu().numpy()        

In [6]:
def compute_tfidf(X):
    """
    Compute TF-IDF for a term-document value matrix.
    Parameters:
    X: np.ndarray (num_terms, num_documents)
    Returns:
    tfidf_matrix: np.ndarray (num_terms, num_documents) - TF-IDF values
    """
    # Compute Term Frequency (TF) - Normalize by column sum
    tf = X / np.sum(X, axis=0, keepdims=True)
    tf[np.isnan(tf)] = 0  # Handle division by zero
    # Compute Document Frequency (DF) - Count nonzero occurrences of each term
    df = np.count_nonzero(X, axis=1)
    # Compute Inverse Document Frequency (IDF) - Log-scaled
    num_documents = X.shape[1]
    idf = np.log((num_documents + 1) / (df + 1)) + 1  # Smoothing
    # Compute TF-IDF
    tfidf_matrix = tf * idf[:, np.newaxis]
    return tfidf_matrix


train_tag_to_idx = {t: i for i, t in enumerate(tags)}

pti = train_tag_item_counts.copy()
pti.data /= pti.data.sum()

characteristic_neuron_per_tag_dict = {}
top_neuron_per_tag_dict = {}

for base_model_name in cf_models:
    print(base_model_name)
    sparse_item_embeddings = sparse_item_embedding_dict[base_model_name]

    tag_neuron_activity = pti @ sparse_item_embeddings  # tag x neuron (CSR matrix)

    # neuron -> tag that elicits most distinctive response in this neuron
    top_tag_per_neuron = tags[compute_tfidf(tag_neuron_activity.toarray().T).argmax(axis=1)]  # term = neuron, document = tag
    top_tag_per_neuron[~neuron_is_alive] = None
    print(pl.Series("top_tag", top_tag_per_neuron[neuron_is_alive]).value_counts(sort=True).head(5))

    # neuron -> tag that best characterizes the neuron's overall activity
    characteristic_tag_per_neuron = tags[compute_tfidf(tag_neuron_activity.toarray()).argmax(axis=0)]  # term = tag, document = neuron
    characteristic_tag_per_neuron[~neuron_is_alive] = None
    print(pl.Series("characteristic_tag", characteristic_tag_per_neuron[neuron_is_alive]).value_counts(sort=True).head(5))

    # tag -> neuron that most representatively encodes this tag
    characteristic_neuron_per_tag = compute_tfidf(tag_neuron_activity.toarray().T).argmax(axis=0)  # term = neuron, document = tag
    print(pl.Series("characteristic_neuron", characteristic_neuron_per_tag).value_counts(sort=True).head(5))

    # tag -> neuron whose firing is most unique to that tag
    top_neuron_per_tag = compute_tfidf(tag_neuron_activity.toarray()).argmax(axis=1)  # term = neuron, document = tag
    print(pl.Series("top_neuron", top_neuron_per_tag).value_counts(sort=True).head(5))

    characteristic_neuron_per_tag_dict[base_model_name] = characteristic_neuron_per_tag
    top_neuron_per_tag_dict[base_model_name] = top_neuron_per_tag


characteristic_neuron_per_tag_dict["TEASER"] = np.arange(len(tags))
top_neuron_per_tag_dict["TEASER"] = np.arange(len(tags))

ELSA


/tmp/ipykernel_550078/2780223166.py:10: RuntimeWarning: invalid value encountered in divide
  tf = X / np.sum(X, axis=0, keepdims=True)


shape: (5, 2)
┌──────────────┬───────┐
│ top_tag      ┆ count │
│ ---          ┆ ---   │
│ str          ┆ u32   │
╞══════════════╪═══════╡
│ classic rock ┆ 536   │
│ french house ┆ 10    │
│ mistagged    ┆ 10    │
│ Coldplay     ┆ 9     │
│ ninja tune   ┆ 9     │
└──────────────┴───────┘
shape: (5, 2)
┌────────────────────┬───────┐
│ characteristic_tag ┆ count │
│ ---                ┆ ---   │
│ str                ┆ u32   │
╞════════════════════╪═══════╡
│ classic rock       ┆ 553   │
│ rock               ┆ 275   │
│ indie              ┆ 161   │
│ electronic         ┆ 101   │
│ pop                ┆ 69    │
└────────────────────┴───────┘
shape: (5, 2)
┌───────────────────────┬───────┐
│ characteristic_neuron ┆ count │
│ ---                   ┆ ---   │
│ i64                   ┆ u32   │
╞═══════════════════════╪═══════╡
│ 1866                  ┆ 10    │
│ 7016                  ┆ 10    │
│ 1765                  ┆ 9     │
│ 6800                  ┆ 8     │
│ 2985                  ┆ 8     │
└─

In [7]:
from util import evaluate_ndcg_at_k, evaluate_recall_at_k


def build_tag_holdout_split(
    user_item_csr: sp.csr_matrix, tag_df: pl.DataFrame, items: np.ndarray, target_ratio: float = 0.2, min_user_tag_support: int = 2, max_user_tag_support: int = 10
) -> tuple[sp.csr_matrix, sp.csr_matrix, np.ndarray]:
    base_inputs, base_targets = split_input_target_interactions(user_item_csr, target_ratio)

    tags, tag_item_counts = get_tag_item_counts(tag_df)
    tag_item_counts = tag_item_counts.copy().tocsr()
    tag_item_counts.data[:] = 1
    item_tag_counts = tag_item_counts.T.tocsr()

    input_rows = []
    target_rows = []
    selected_tags = []
    tag_counts = []

    for user_idx in range(base_inputs.shape[0]):
        input_row = base_inputs.getrow(user_idx)
        target_row = base_targets.getrow(user_idx)
        input_item_indices = input_row.indices
        target_item_indices = target_row.indices

        if len(input_item_indices) == 0:
            continue

        user_tag_counts = np.asarray(item_tag_counts[input_item_indices].sum(axis=0)).ravel()
        eligible_tags = np.where(
            (user_tag_counts >= min_user_tag_support) &
            (user_tag_counts <= max_user_tag_support)
        )[0]
        if len(eligible_tags) == 0:
            print("No tags for user "+str(user_idx))
            continue

        selected_tag = np.random.choice(eligible_tags)
        tag_items = set(tag_item_counts.getrow(selected_tag).indices.tolist())

        tagCount = user_tag_counts[selected_tag]

        moved_from_input = np.array([i for i in input_item_indices if i in tag_items], dtype=np.int32)
        kept_in_input = np.array([i for i in input_item_indices if i not in tag_items], dtype=np.int32)
        new_target_items = np.concatenate([target_item_indices, moved_from_input]).astype(np.int32)

        if len(kept_in_input) == 0 or len(new_target_items) == 0:
            print("Nothing in input "+str(user_idx))
            continue

        input_data = np.ones(len(kept_in_input), dtype=np.float32)
        target_data = np.ones(len(new_target_items), dtype=np.float32)
        input_rows.append(sp.csr_matrix((input_data, ([0] * len(kept_in_input), kept_in_input)), shape=(1, user_item_csr.shape[1])))
        target_rows.append(sp.csr_matrix((target_data, ([0] * len(new_target_items), new_target_items)), shape=(1, user_item_csr.shape[1])))
        selected_tags.append(selected_tag)
        tag_counts.append(tagCount)

    inputs = sp.vstack(input_rows)
    targets = sp.vstack(target_rows)
    return inputs, targets, np.array(selected_tags, dtype=np.int64), np.array(tag_counts, dtype=np.int64)


def evaluate_steering(
    model, inputs: sp.csr_matrix, targets: sp.csr_matrix, segment_inputs: np.ndarray, tag_df: pl.DataFrame, device: torch.device, k: int = 20, batch_size: int = 1024
) -> dict:
    tags, tag_item_counts = get_tag_item_counts(tag_df)
    tag_item_counts.data[:] = 1
    segment_targets = tag_item_counts[segment_inputs]

    personal_inputs = Dataloader(inputs, batch_size, device)
    steering_inputs = Dataloader(segment_inputs, batch_size, device)
    personal_targets = Dataloader(targets, batch_size, device)
    segment_targets = Dataloader(segment_targets, batch_size, device)

    # Personal targets (binary)
    personal_target_matrix = targets.copy().tocsr()
    personal_target_matrix.data[:] = 1
    
    # Segment targets (binary)
    segment_targets = tag_item_counts[segment_inputs].tocsr()
    segment_targets.data[:] = 1
    
    # personalSegment = intersection(personal, segment)
    personal_segment_targets = personal_target_matrix.multiply(segment_targets).tocsr()
    personal_segment_targets.data[:] = 1
    personal_segment_targets.eliminate_zeros()
    
    # personalNotSegment = personal \ segment
    personal_not_segment_targets = (personal_target_matrix - personal_segment_targets).tocsr()
    personal_not_segment_targets.eliminate_zeros()
    if personal_not_segment_targets.nnz > 0:
        personal_not_segment_targets.data[:] = 1

    
    personal_targets_dl = Dataloader(personal_target_matrix, batch_size, device)
    segment_targets_dl = Dataloader(segment_targets, batch_size, device)
    personal_segment_targets_dl = Dataloader(personal_segment_targets, batch_size, device)
    personal_not_segment_targets_dl = Dataloader(personal_not_segment_targets, batch_size, device)
    
    model.eval()
    
    personal_recalls = evaluate_recall_at_k(model, zip(personal_inputs, steering_inputs), personal_targets_dl, k)
    segment_recalls = evaluate_recall_at_k(model, zip(personal_inputs, steering_inputs), segment_targets_dl, k)
    
    personal_ndcgs = evaluate_ndcg_at_k(model, zip(personal_inputs, steering_inputs), personal_targets_dl, k)
    segment_ndcgs = evaluate_ndcg_at_k(model, zip(personal_inputs, steering_inputs), segment_targets_dl, k)
    
    personal_segment_recalls = evaluate_recall_at_k(model, zip(personal_inputs, steering_inputs), personal_segment_targets_dl, k)
    personal_segment_ndcgs = evaluate_ndcg_at_k(model, zip(personal_inputs, steering_inputs), personal_segment_targets_dl, k)
    
    personal_not_segment_recalls = evaluate_recall_at_k(model, zip(personal_inputs, steering_inputs), personal_not_segment_targets_dl, k)
    personal_not_segment_ndcgs = evaluate_ndcg_at_k(model, zip(personal_inputs, steering_inputs), personal_not_segment_targets_dl, k)
    


    results = {
        f"recall@{k}(personal)": {
            "mean": float(np.mean(personal_recalls)),
            "se": float(np.std(personal_recalls) / np.sqrt(len(personal_recalls)))
        },
        f"ndcg@{k}(personal)": {
            "mean": float(np.mean(personal_ndcgs)),
            "se": float(np.std(personal_ndcgs) / np.sqrt(len(personal_ndcgs)))
        },
    
        f"recall@{k}(segment)": {
            "mean": float(np.mean(segment_recalls)),
            "se": float(np.std(segment_recalls) / np.sqrt(len(segment_recalls)))
        },
        f"ndcg@{k}(segment)": {
            "mean": float(np.mean(segment_ndcgs)),
            "se": float(np.std(segment_ndcgs) / np.sqrt(len(segment_ndcgs)))
        },
    
        f"recall@{k}(personalSegment)": {
            "mean": float(np.mean(personal_segment_recalls)),
            "se": float(np.std(personal_segment_recalls) / np.sqrt(len(personal_segment_recalls)))
        },
        f"ndcg@{k}(personalSegment)": {
            "mean": float(np.mean(personal_segment_ndcgs)),
            "se": float(np.std(personal_segment_ndcgs) / np.sqrt(len(personal_segment_ndcgs)))
        },
    
        f"recall@{k}(personalNotSegment)": {
            "mean": float(np.mean(personal_not_segment_recalls)),
            "se": float(np.std(personal_not_segment_recalls) / np.sqrt(len(personal_not_segment_recalls)))
        },
        f"ndcg@{k}(personalNotSegment)": {
            "mean": float(np.mean(personal_not_segment_ndcgs)),
            "se": float(np.std(personal_not_segment_ndcgs) / np.sqrt(len(personal_not_segment_ndcgs)))
        },
    }
    return results


test_inputs, test_targets, test_segment_inputs, tag_counts = build_tag_holdout_split(
    test_csr, tag_df, items, target_ratio=elsa_cfg["target_interaction_ratio"], min_user_tag_support=MIN_USER_TAG_SUPPORT, max_user_tag_support=MAX_USER_TAG_SUPPORT
)
print(
    f"Tag-holdout-80/20 evaluation users={test_inputs.shape[0]}, min_user_tag_support={MIN_USER_TAG_SUPPORT}, max_user_tag_support={MAX_USER_TAG_SUPPORT}, "
    f"target_tag_sampling=random, mean input nnz={test_inputs.nnz / test_inputs.shape[0]:.2f}, "
    f"mean target nnz={test_targets.nnz / test_targets.shape[0]:.2f}"
)


No tags for user 8
No tags for user 82
No tags for user 158
No tags for user 221
No tags for user 242
No tags for user 264
No tags for user 565
No tags for user 616
No tags for user 762
No tags for user 825
No tags for user 967
No tags for user 1214
No tags for user 1327
No tags for user 1790
No tags for user 2085
No tags for user 2287
No tags for user 2292
No tags for user 2345
No tags for user 2430
No tags for user 2577
No tags for user 2668
No tags for user 2830
No tags for user 2859
No tags for user 2923
No tags for user 2947
No tags for user 3020
No tags for user 3248
No tags for user 3533
No tags for user 3623
No tags for user 3668
No tags for user 3879
No tags for user 4091
No tags for user 4179
No tags for user 4182
No tags for user 4399
No tags for user 4481
No tags for user 4561
No tags for user 4578
No tags for user 4760
No tags for user 4939
No tags for user 4978
No tags for user 5026
No tags for user 5031
No tags for user 5089
No tags for user 5183
Nothing in input 5266
No

In [8]:
import pandas as pd
pd.Series(tag_counts).describe()

count    56690.000000
mean         8.792821
std          6.310155
min          5.000000
25%          5.000000
50%          7.000000
75%         10.000000
max        100.000000
dtype: float64

In [9]:
results = {}
for base_model_name in cf_models:
    print(base_model_name)
    results[base_model_name] = {"map: best characterizes": {}, "map: most unique": {}}
    cf_model = cf_models[base_model_name]
    sae = sae_models[base_model_name]
    characteristic_neuron_per_tag = characteristic_neuron_per_tag_dict[base_model_name]
    top_neuron_per_tag = top_neuron_per_tag_dict[base_model_name]
    for alpha in tqdm(np.linspace(0, 0.3, 7)):
        res = results[base_model_name]["map: best characterizes"]
        a = float(alpha)
        m = SAESteeredModel(base_model=cf_model, sae=sae, concept_neuron_mapping=torch.tensor(characteristic_neuron_per_tag, dtype=int, device=device), alpha=a)
        res[a] = evaluate_steering(m, test_inputs, test_targets, test_segment_inputs, tag_df, device)
    for alpha in tqdm(np.linspace(0, 0.3, 7)):
        res = results[base_model_name]["map: most unique"]
        a = float(alpha)
        m = SAESteeredModel(base_model=cf_model, sae=sae, concept_neuron_mapping=torch.tensor(top_neuron_per_tag, dtype=int, device=device), alpha=a)
        res[a] = evaluate_steering(m, test_inputs, test_targets, test_segment_inputs, tag_df, device)

for base_model_name, cf_model in direct_models.items():
    print(base_model_name)
    results[base_model_name] = {"map: direct tag": {}}
    for alpha in tqdm(np.linspace(0, 0.3, 7)):
        res = results[base_model_name]["map: direct tag"]
        a = float(alpha)
        m = TEASERSteeredModel(base_model=cf_model, alpha=a)
        res[a] = evaluate_steering(m, test_inputs, test_targets, test_segment_inputs, tag_df, device)



        


results

ELSA


  0%|          | 0/7 [01:47<?, ?it/s]


KeyboardInterrupt: 

In [10]:
for base_model_name, cf_model in direct_models.items():
    print(base_model_name)
    results[base_model_name] = {"map: direct tag": {}}
    for alpha in tqdm(np.linspace(0, 0.3, 7)):
        res = results[base_model_name]["map: direct tag"]
        a = float(alpha)
        m = TEASERSteeredModel(base_model=cf_model, alpha=a)
        res[a] = evaluate_steering(m, test_inputs, test_targets, test_segment_inputs, tag_df, device)



TEASER


  0%|          | 0/7 [00:04<?, ?it/s]


IndexError: index 1754 is out of bounds for dimension 1 with size 1753

In [ ]:
import pickle

with open("results_steering_MSD.pkl", "wb") as f:
    pickle.dump(results, f)

In [ ]:
import pickle

with open("results_steering_MSD.pkl", "rb") as f:
    results = pickle.load(f)

In [ ]:
results

In [ ]:
import matplotlib.pyplot as plt


def plot(data: dict[str, dict], metric_x: str, metric_y: str, path = None):
    fig, ax = plt.subplots(figsize=(4, 3))

    # Iterate over variants and plot each with its own color
    for variant, res in data.items():
        params = sorted(res.keys())
        x = [res[p][metric_x]["mean"] for p in params]
        y = [res[p][metric_y]["mean"] for p in params]
        xerr = [res[p][metric_x]["se"] for p in params]
        yerr = [res[p][metric_y]["se"] for p in params]

        line = ax.errorbar(x, y, xerr=xerr, yerr=yerr, fmt="o-", lw=1, capsize=2, markersize=4, label=variant)

        # Add parameter labels
        color = line[0].get_color()
        for i, p in enumerate(params):
            ax.text(x[i], y[i], f"{round(p, 3)}", fontsize=7, va="bottom", ha="left", color=color)

    # Axis setup
    ax.set_xlabel(metric_x)
    ax.set_ylabel(metric_y)
    ax.grid(True, ls="--", lw=0.4, alpha=0.6)
    ax.set_xlim(0.0, 0.45)
    ax.set_ylim(0.0, 0.45)
    # ax.set_aspect("equal", adjustable="box")

    ax.legend(frameon=False, fontsize=8)
    fig.tight_layout()

    if path:
        fig.savefig(path, dpi=600, bbox_inches="tight")

    plt.show()

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D


def _fmt_alpha(a):
    # Pretty alpha label: 0.3 instead of 0.30000000000000004
    s = f"{float(a):.3f}".rstrip("0").rstrip(".")
    return s if s else "0"


def plot_results(
    data: dict,
    metric_x: str,
    metric_y: str,
    path = None,
    xlim=(0.0, 0.65),
    ylim=(0.0, 0.40),
    annotate=True,
):
    """
    data:
      { base_model: {
          "map: best characterizes": { alpha: {metric_x:{mean,se}, metric_y:{mean,se}, ...}, ... },
          "map: most unique":        { alpha: {...}, ... }
        }, ... }
    """

    color_by_model = {"ELSA": "tab:orange", "MultVAE": "tab:blue"}
    style_by_map = {
        "map: best characterizes": dict(linestyle="-", marker="o"),
        "map: most unique": dict(linestyle="--", marker="s"),
    }

    fig, ax = plt.subplots(figsize=(5, 3))

    for base_model, maps in data.items():
        color = color_by_model.get(base_model, None)
        for map_type, series in maps.items():
            if not series:
                continue
            style = style_by_map.get(map_type, dict(linestyle="-", marker="o"))

            # Sort alphas numerically
            alphas = sorted(series.keys(), key=float)

            # Extract, skipping points missing either metric
            x, y, xerr, yerr, alphas_kept = [], [], [], [], []
            for a in alphas:
                rec = series[a]
                if metric_x in rec and metric_y in rec:
                    x.append(rec[metric_x]["mean"])
                    y.append(rec[metric_y]["mean"])
                    xerr.append(rec[metric_x].get("se", 0.0))
                    yerr.append(rec[metric_y].get("se", 0.0))
                    alphas_kept.append(a)

            if not x:
                continue

            line = ax.errorbar(x, y, xerr=xerr, yerr=yerr, lw=1, capsize=2, markersize=4, color=color, label=f"{base_model} • {map_type}", **style)

            if annotate:
                tcolor = color if color is not None else line[0].get_color()
                for xi, yi, a in zip(x, y, alphas_kept):
                    ax.text(xi, yi, f" {_fmt_alpha(a)}", fontsize=7, va="bottom", ha="left", color=tcolor)

    # Axes & layout
    ax.set_xlabel(metric_x)
    ax.set_ylabel(metric_y)
    ax.grid(True, ls="--", lw=0.4, alpha=0.6)
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    # ax.set_aspect("equal", adjustable="box")  # uncomment for equal scaling

    # Two legends: colors (base models) and styles (map types)
    color_handles = [Line2D([0], [0], color=c, lw=2, label=m) for m, c in color_by_model.items()]
    style_handles = [Line2D([0], [0], color="black", lw=2, linestyle=st["linestyle"], marker=st["marker"], label=mt) for mt, st in style_by_map.items()]
    leg1 = ax.legend(handles=color_handles, title="Base model", frameon=False, fontsize=8, loc="upper left")
    ax.add_artist(leg1)
    ax.legend(handles=style_handles, title="Map type", frameon=False, fontsize=8, loc="upper right")

    fig.tight_layout()
    if path:
        fig.savefig(path, dpi=600, bbox_inches="tight")  # or .pdf/.svg for vector graphics
    plt.show()

In [ ]:
plot_results(data=results, metric_x="ndcg@20(segment)", metric_y="ndcg@20(personal)", path="results/figures/steering_MSD.png")